In [1]:
# Import libraries
import tensorflow as tf
from tensorflow.keras.layers.experimental import preprocessing

import numpy as np
import os
import time

2022-11-28 13:12:25.489756: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2022-11-28 13:12:26.795912: E tensorflow/stream_executor/cuda/cuda_blas.cc:2981] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2022-11-28 13:12:30.284675: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; dlerror: libnvinfer.so.7: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: :/home/mathidiot/miniconda3/envs/tf/lib/
2022-11-28 13:12:30.285092: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer_plugin.so.7'; dlerror: libnvinfer_pl

In [23]:
# Load file data
text = ''
silmarillion = open('./text_files/tolkien/silmarillion.txt', 'rb').read().decode(encoding='utf-8')
hobbit =       open('./text_files/tolkien/hobbit.txt', 'rb').read().decode(encoding='utf-8')
lotr1  =       open('./text_files/tolkien/lotr1.txt', 'rb').read().decode(encoding='utf-8')
lotr2  =       open('./text_files/tolkien/lotr2.txt', 'rb').read().decode(encoding='utf-8')
lotr3  =       open('./text_files/tolkien/lotr3.txt', 'rb').read().decode(encoding='utf-8')
text += silmarillion + hobbit + lotr1 + '\n' + lotr2 + '\n' + lotr3

print('Length of text: {} characters'.format(len(text)))

Length of text: 3726832 characters


In [25]:
# Verify the first part of our data
print(text[:200])


AINULINDALË
The Music of the Ainur
There was Eru, the One, who in Arda is called Ilúvatar; and he made first the
Ainur, the Holy Ones, that were the offspring of his thought, and they were with
him b


In [24]:
# Now we'll get a list of the unique characters in the file. This will form the
# vocabulary of our ne200twork. There may be some characters we want to remove from this 
# set as we refine the network.
vocab = sorted(set(text))
print('{} unique characters'.format(len(vocab)))
print(vocab)

117 unique characters
['\n', ' ', '!', '"', '$', '%', "'", '(', ')', '*', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '<', '>', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '\\', ']', '^', '_', '`', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '~', '£', '§', '«', '¬', '»', 'Ä', 'É', 'Ê', 'Ë', 'Ú', 'á', 'â', 'ä', 'é', 'ê', 'ë', 'í', 'î', 'ó', 'ô', 'ö', 'ú', 'û', '–', '—', '‘', '’', '“', '”', '•']


In [26]:
# Next, we'll encode encode these characters into numbers so we can use them
# with our neural network, then we'll create some mappings between the characters
# and their numeric representations
ids_from_chars = preprocessing.StringLookup(vocabulary=list(vocab))
chars_from_ids = tf.keras.layers.experimental.preprocessing.StringLookup(vocabulary=ids_from_chars.get_vocabulary(), invert=True)

# Here's a little helper function that we can use to turn a sequence of ids
# back into a string:path_to_file
# turn them into a string:
def text_from_ids(ids):
  joinedTensor = tf.strings.reduce_join(chars_from_ids(ids), axis=-1)
  return joinedTensor.numpy().decode("utf-8")

In [7]:
# Now we'll verify that they work, by getting the code for "A", and then looking
# that up in reverse
testids = ids_from_chars(["T", "r", "u", "t", "h"])
testids

<tf.Tensor: shape=(5,), dtype=int64, numpy=array([41, 64, 67, 66, 54])>

In [8]:
chars_from_ids(testids)

<tf.Tensor: shape=(5,), dtype=string, numpy=array([b'T', b'r', b'u', b't', b'h'], dtype=object)>

In [9]:
testString = text_from_ids( testids )
testString

'Truth'

In [10]:
# First, create a stream of encoded integers from our text
all_ids = ids_from_chars(tf.strings.unicode_split(text, 'UTF-8'))
all_ids

<tf.Tensor: shape=(631932,), dtype=int64, numpy=array([ 1, 22, 30, ..., 65,  8,  1])>

In [11]:
# Now, convert that into a tensorflow dataset
ids_dataset = tf.data.Dataset.from_tensor_slices(all_ids)

In [27]:
# Finally, let's batch these sequences up into chunks for our training
seq_length = 100
sequences = ids_dataset.batch(seq_length+1, drop_remainder=True)

# This function will generate our sequence pairs:
def split_input_target(sequence):
    input_text = sequence[:-1]
    target_text = sequence[1:]
    return input_text, target_text

# Call the function for every sequence in our list to create a new dataset
# of input->target pairs
dataset = sequences.map(split_input_target)

In [28]:
# Verify our sequences
for input_example, target_example in  dataset.take(1):
    print("Input: ", text_from_ids(input_example))
    print("--------")
    print("Target: ", text_from_ids(target_example))

Input:  
7AFMDAF:7Do
LYV EgeZT aW fYV 7Z`gd
LYVdV iRe ;dg% fYV G`V% iYa Z` 7dUR Ze TR^^VU A^£hRfRd5 R`U YV _
--------
Target:  7AFMDAF:7Do
LYV EgeZT aW fYV 7Z`gd
LYVdV iRe ;dg% fYV G`V% iYa Z` 7dUR Ze TR^^VU A^£hRfRd5 R`U YV _R


In [29]:
# Finally, we'll randomize the sequences so that we don't just memorize the books
# in the order they were written, then build a new streaming dataset from that.
# Using a streaming dataset allows us to pass the data to our network bit by bit,
# rather than keeping it all in memory. We'll set it to figure out how much data
# to prefetch in the background.

BATCH_SIZE = 64
BUFFER_SIZE = 10000

dataset = (
    dataset
    .shuffle(BUFFER_SIZE)
    .batch(BATCH_SIZE, drop_remainder=True)
    .prefetch(tf.data.experimental.AUTOTUNE))

dataset

<PrefetchDataset element_spec=(TensorSpec(shape=(64, 100), dtype=tf.int64, name=None), TensorSpec(shape=(64, 100), dtype=tf.int64, name=None))>

In [30]:
# Create our custom model. Given a sequence of characters, this
# model's job is to predict what character should come next.
class AustenTextModel(tf.keras.Model):

  # This is our class constructor method, it will be executed when
  # we first create an instance of the class 
  def __init__(self, vocab_size, embedding_dim, rnn_units):
    super().__init__(self)

    # Our model will have three layers:
    
    # 1. An embedding layer that handles the encoding of our vocabulary into
    #    a vector of values suitable for a neural network
    self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)

    # 2. A GRU layer that handles the "memory" aspects of our RNN. If you're
    #    wondering why we use GRU instead of LSTM, and whether LSTM is better,
    #    take a look at this article: https://datascience.stackexchange.com/questions/14581/when-to-use-gru-over-lstm
    #    then consider trying out LSTM instead (or in addition to!)
    self.gru = tf.keras.layers.GRU(rnn_units, return_sequences=True, return_state=True)

    # 3. Our output layer that will give us a set of probabilities for each
    #    character in our vocabulary.
    self.dense = tf.keras.layers.Dense(vocab_size)

  # This function will be executed for each epoch of our training. Here
  # we will manually feed information from one layer of our network to the 
  # next.
  def call(self, inputs, states=None, return_state=False, training=False):
    x = inputs

    # 1. Feed the inputs into the embedding layer, and tell it if we are
    #    training or predicting
    x = self.embedding(x, training=training)

    # 2. If we don't have any state in memory yet, get the initial random state
    #    from our GRUI layer.
    if states is None:
      states = self.gru.get_initial_state(x)
    
    # 3. Now, feed the vectorized input along with the current state of memory
    #    into the gru layer.
    x, states = self.gru(x, initial_state=states, training=training)

    # 4. Finally, pass the results on to the dense layer
    x = self.dense(x, training=training)

    # 5. Return the results
    if return_state:
      return x, states
    else: 
      return x

In [31]:
# Create an instance of our model
vocab_size=len(ids_from_chars.get_vocabulary())
embedding_dim = 256
rnn_units = 1024

model = AustenTextModel(vocab_size, embedding_dim, rnn_units)

In [32]:
# Verify the output of our model is correct by running one sample through
# This will also compile the model for us. This step will take a bit.
for input_example_batch, target_example_batch in dataset.take(1):
    example_batch_predictions = model(input_example_batch)
    print(example_batch_predictions.shape, "# (batch_size, sequence_length, vocab_size)")


(64, 100, 118) # (batch_size, sequence_length, vocab_size)


In [18]:
# Now let's view the model summary
model.summary()

Model: "austen_text_model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       multiple                  24320     
                                                                 
 gru (GRU)                   multiple                  3938304   
                                                                 
 dense (Dense)               multiple                  97375     
                                                                 
Total params: 4,059,999
Trainable params: 4,059,999
Non-trainable params: 0
_________________________________________________________________


In [34]:
loss = tf.losses.SparseCategoricalCrossentropy(from_logits=True)
model.compile(optimizer='adam', loss=loss)

In [35]:
history = model.fit(dataset, epochs=20)

Epoch 1/20
 1/97 [..............................] - ETA: 8:39 - loss: 4.7712

2022-11-21 13:48:58.084802: W tensorflow/core/framework/cpu_allocator_impl.cc:82] Allocation of 26214400 exceeds 10% of free system memory.
2022-11-21 13:48:58.094985: W tensorflow/core/framework/cpu_allocator_impl.cc:82] Allocation of 26214400 exceeds 10% of free system memory.


97/97 [==============================] - 76s 740ms/step - loss: 2.8547
Epoch 2/20
97/97 [==============================] - 74s 744ms/step - loss: 2.0838
Epoch 3/20
97/97 [==============================] - 75s 754ms/step - loss: 1.8238
Epoch 4/20
97/97 [==============================] - 75s 752ms/step - loss: 1.6106
Epoch 5/20
97/97 [==============================] - 75s 751ms/step - loss: 1.4565
Epoch 6/20
10/97 [==>...........................] - ETA: 1:05 - loss: 1.3996

KeyboardInterrupt: 

In [36]:
# Here's the code we'll use to sample for us. It has some extra steps to apply
# the temperature to the distribution, and to make sure we don't get empty
# characters in our text. Most importantly, it will keep track of our model
# state for us.

class OneStep(tf.keras.Model):
  def __init__(self, model, chars_from_ids, ids_from_chars, temperature=1.0):
    super().__init__()
    self.temperature=temperature
    self.model = model
    self.chars_from_ids = chars_from_ids
    self.ids_from_chars = ids_from_chars

    # Create a mask to prevent "" or "[UNK]" from being generated.
    skip_ids = self.ids_from_chars(['','[UNK]'])[:, None]
    sparse_mask = tf.SparseTensor(
        # Put a -inf at each bad index.
        values=[-float('inf')]*len(skip_ids),
        indices = skip_ids,
        # Match the shape to the vocabulary
        dense_shape=[len(ids_from_chars.get_vocabulary())]) 
    self.prediction_mask = tf.sparse.to_dense(sparse_mask,validate_indices=False)

  @tf.function
  def generate_one_step(self, inputs, states=None):
    # Convert strings to token IDs.
    input_chars = tf.strings.unicode_split(inputs, 'UTF-8')
    input_ids = self.ids_from_chars(input_chars).to_tensor()

    # Run the model.
    # predicted_logits.shape is [batch, char, next_char_logits] 
    predicted_logits, states =  self.model(inputs=input_ids, states=states, 
                                          return_state=True)
    # Only use the last prediction.
    predicted_logits = predicted_logits[:, -1, :]
    predicted_logits = predicted_logits/self.temperature
    
    # Apply the prediction mask: prevent "" or "[UNK]" from being generated.
    predicted_logits = predicted_logits + self.prediction_mask

    # Sample the output logits to generate token IDs.
    predicted_ids = tf.random.categorical(predicted_logits, num_samples=1)
    predicted_ids = tf.squeeze(predicted_ids, axis=-1)

    # Return the characters and model state.
    return chars_from_ids(predicted_ids), states


In [37]:
# Create an instance of the character generator
one_step_model = OneStep(model, chars_from_ids, ids_from_chars)

# Now, let's generate a 1000 character chapter by giving our model "Chapter 1"
# as its starting text
states = None
next_char = tf.constant(['Chapter 1'])
result = [next_char]

for n in range(1000):
  next_char, states = one_step_model.generate_one_step(next_char, states=states)
  result.append(next_char)

result = tf.strings.join(result)

# Print the results formatted.
print(result[0].numpy().decode('utf-8'))




Chapter 1ZUYe( 8gf fYV FR_RX
EV^ZRf^ YR`U Saaf YZ^^% Rf fYVk T^RgXYfVde( 8gf aW R^_Vefda`U Sgf 7dXR`( 7f Zf iVdV WZfVdagdVU fYRf `a dagd`VUVU Z` GdTYg^U%
R`U eagXYf fYVdV fYRf fYVk _V^]% ZW ^ahV R`U fiVdV fYVdV5 R`U YZeYVU R_a`X(
7`UZ^ efdgVWfYVU Rf ^agXYf RWfVd fYV_( 7WfVd Wad gb fYV fadVe aW LRgda` R`U eVV R`TVdeR`f XdVRfVdTR YV SVeVVf YZ_5 R`U fYV YRhV efRdeV
?£dZ` URd]`Vee4 Gd eYZbe R`U ;^dReYVd% Sgf efdaiVU XdVRbVdk YZ_ XdaeW Vde_VVU% eYaiV^k `VRd
eYR^^ ea^U Rf ^ZefV` R^ZeV YZ^^VU( 7e fYV NR^Rd WR_VeYVU _ZeYag` Sk iYZ^V fa eYR^^ SVYV fYV iZ^U5 Sgf
YV iadV5 R`U fYV CZ`X aW fYV Fa^Uad
YV iag^U R`UiRdUV% iZfY YV iRe fa
_R( Fai`Z` YZe iV`fgdZ`X% fYV
bdabfeVe( 8V YV XdVRf RiRk% YZe iVdV gb R XdRhV`f ad YZe WV^^agfY( 8gf fYV ^R`U aW 8^adUa^ZU _RUV eYR^^ Yae aW 7d_RdR^W'YR`Ze% Wda_ fYV iad]e aW ;s iZfV ReeVVU
YZe
YRhV iZfY b^RkVe
baiVde% RefVd% _Vefa _aZ`VR `a SZjf^V fdg`Vi kYR` fYaeV RXRZ`ef _aef^Vee
eYV iVdV gbR`ef5»
eVRf% Wad fa YZ_ZV` fYV eVTdVfZd aW LZdZSagd( OYV ^R`U TR_V ad YR^^e(